# 02.6 -- Does curvature through a *trained plain* decoder recover a known answer?

Phase 02.6 screens free candidate decoder substrates for local mean curvature. The plain
auto-encoder is the first of the two free candidates (the second, trained under
`topoae.train_topoae`, is `02.6_swiss_roll_topoae_curvature_check.ipynb`). This notebook asks
whether curvature taken exactly through a trained `cae.PlainAutoEncoder` decoder recovers the
Swiss roll's closed-form answer.

**Why the existing Swiss roll notebooks do not already answer this.**
`02.2_swiss_roll_cae_check.ipynb` and `02.4_swiss_roll_topoae_check.ipynb` both use
`PlainAutoEncoder` as their matched baseline, and it passes there -- but both notebooks test
**reconstruction only**. Nothing in this repository has ever tested this decoder's **curvature**,
and `02.5-NOTE-substrate-selection.md` Section 4 states flatly that reconstruction quality
validates nothing about curvature. Reading the existing reconstruction pass as evidence about
curvature is `02.6-RESEARCH.md`'s Pitfall 1 exactly. This notebook closes that gap for one
candidate.

This notebook gates nothing and seals nothing. It is a CLAUDE.md-mandated sanity check, not the
phase's screening measurement -- that is plan `02.6-05`'s four-seed runner, scored against the
bars ratified in `02.6-SCREENING-RULE.md`.

## §1. Setup

In [ ]:
import sys
import time
from pathlib import Path

# import pu_manifold exactly as the other notebooks do -- relative, never from src/effdim/
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import matplotlib.pyplot as plt
import numpy as np
import torch

from pu_manifold import cae, chart_curvature, curvature_probe, decoder_curvature

SEED = 0                  # torch / split seed
FIXTURE_SEED = 20260807   # the roll's own random_state; identical across phase 02.5 and 02.6
N_POINTS = 3000
LATENT_DIM = 2             # the Swiss roll's true intrinsic dimension
HIDDEN = (64, 64, 64)
K_BASELINE = 30            # the raw-point comparator's neighbourhood size

print(f"torch {torch.__version__}  numpy {np.__version__}")
print(f"cae               : {cae.__file__}")
print(f"chart_curvature   : {chart_curvature.__file__}")
print(f"curvature_probe   : {curvature_probe.__file__}")
print(f"decoder_curvature : {decoder_curvature.__file__}")
print(f"convention        : decoder_curvature={decoder_curvature.CURVATURE_CONVENTION!r}  "
      f"curvature_probe={curvature_probe.CURVATURE_CONVENTION!r}")


## §2. The Swiss roll, and its curvature in closed form

`curvature_probe.make_swiss_roll_fixture` is called rather than transcribed, matching CLAUDE.md's
required preprocessing: `sklearn.datasets.make_swiss_roll` with `noise=0.0` and a fixed
`random_state`, centred, then divided by **one global scalar** standard deviation -- a single
number, so the shape is preserved and only the overall size changes.

`FIXTURE_SEED = 20260807` is the identical draw the CAE negative control (`02.5-09`) and plan
`02.6-05`'s runner use, so this candidate's numbers are directly comparable to that negative
control's and to the runner's own four-seed table.

The analytic mean curvature **vector** comes from `decoder_curvature.swiss_roll_analytic_H_vector`,
hosted in this phase's own module rather than the sealed `curvature_probe.py` (never edited this
milestone). Its row norms are pinned against the sealed module's own `H_norm` by test, and the
pin is re-checked below so nothing about the direction can silently drift from the sealed
magnitude ground truth.

In [ ]:
fx = curvature_probe.make_swiss_roll_fixture(n=N_POINTS, seed=FIXTURE_SEED)
X, t, H_norm, global_std = fx["X"], fx["t"], fx["H_norm"], fx["global_std"]

H_true = decoder_curvature.swiss_roll_analytic_H_vector(t, global_std)
pin = float(np.abs(np.linalg.norm(H_true, axis=1) - H_norm).max())
print(f"X {X.shape}   global_std = {global_std:.4f}")
print(f"analytic ||H|| range [{H_norm.min():.4f}, {H_norm.max():.4f}]")
print(f"derived H vector vs the fixture's own H_norm, max abs diff = {pin:.2e}")

rng = np.random.default_rng(SEED)
perm = rng.permutation(N_POINTS)
n_train = int(0.8 * N_POINTS)
train_idx, holdout_idx = perm[:n_train], perm[n_train:]
x_all = torch.tensor(X, dtype=torch.float32)
x_train, x_holdout = x_all[train_idx], x_all[holdout_idx]
print(f"train {len(train_idx)}   holdout {len(holdout_idx)}")

VIEW = dict(elev=12, azim=-78)  # looks along the roll's extrusion axis, so the spiral shows

fig = plt.figure(figsize=(10, 4.5))
ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=t, cmap="viridis", s=4)
ax.view_init(**VIEW)
ax.set_title("Swiss roll (input), coloured by arc-length t")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax2 = fig.add_subplot(1, 2, 2)
ax2.scatter(X[:, 0], X[:, 2], c=t, cmap="viridis", s=4)
ax2.set_title("same points, x-z plane (the spiral)")
ax2.set_xlabel("x"); ax2.set_ylabel("z"); ax2.set_aspect("equal")
plt.tight_layout(); plt.show()


## §3. Train a plain auto-encoder from scratch

`cae.PlainAutoEncoder(3, 2, hidden=(64, 64, 64), activation="silu")`, trained by
`cae.train_plain_ae`, at the roll's true intrinsic dimension. Every cfg value --
`lr=3e-4, weight_decay=1e-4, batch=64, max_epochs=150` -- is reused **verbatim** from
`notebooks/02.4_swiss_roll_topoae_check.ipynb`, so no new tunable is introduced by a sanity
check. The identical cfg is used by plan `02.6-04`'s TopoAE notebook, so the two free candidates
this phase screens differ only in the topological loss term -- the comparison this phase's
screening rule reads means something only if that holds.

`early_stop_patience` is deliberately left unset: `_train_decoder_protocol` defaults it to
`max_epochs + 1`, so neither this arm nor the TopoAE arm ever early-stops, and both run the
identical 150-epoch budget. Phase 02.4-05's reopened defect was precisely an early-stop asymmetry
between a TopoAE arm and its matched baseline; leaving the budget symmetric here makes that
failure mode unreachable.

In [ ]:
CFG_COMMON = dict(lr=3e-4, weight_decay=1e-4, batch=64, max_epochs=150)

wall0 = time.time()
torch.manual_seed(SEED)
model = cae.PlainAutoEncoder(3, LATENT_DIM, hidden=HIDDEN, activation="silu")
fit = cae.train_plain_ae(model, x_train, dict(seed=SEED, **CFG_COMMON))
model.eval()
train_wall = time.time() - wall0

print(f"plain AE   epochs_run = {fit[\'epochs_run\']:3d}  early_stopped = {fit[\'early_stopped\']}")
print(f"training wall-clock = {train_wall:.1f} s  (fit-reported {fit[\'wallclock_s\']:.1f} s)")
print(f"C2 guard on the trained decoder = {decoder_curvature.assert_c2_decoder(model)!r}")

main_history = [h for h in fit["history"] if h["stage"] == "main"]
plt.figure(figsize=(5, 3))
plt.plot([h["epoch"] for h in main_history], [h["total"] for h in main_history])
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("training loss"); plt.title("plain AE training curve")
plt.tight_layout(); plt.show()


## §4. Reconstruction first -- establishing the premise the next section tests

This section answers only whether the reconstruction is good, and it is reported before any
curvature number on purpose: a low reconstruction error printed beside a poor curvature score is
the finding this notebook exists to be able to show, not a bug to hide
(`02.5-NOTE-substrate-selection.md` Section 4: reconstruction quality validates nothing about
curvature).

**CLAUDE.md's matched-baseline step is degenerate for this candidate, stated here rather than
omitted.** CLAUDE.md names `cae.PlainAutoEncoder` as the matched reconstruction baseline for
reconstruction models -- but the model under test here **is** `cae.PlainAutoEncoder`, so a
reconstruction baseline against itself is vacuous. The substitute used instead is the curvature
comparator introduced in §5-§6: `curvature_probe.centroid_mean_curvature` at `k=30, d=2` on the
identical fixture -- a raw-point estimator that works (Spearman `0.6712` on this fixture,
`02.5-09`) rather than one that is perfect.

In [ ]:
with torch.no_grad():
    y_hold = model(x_holdout)["y"]
    y_all = model(x_all)["y"].numpy()

stats = cae.reconstruction_stats(x_holdout.double(), y_hold.double())
rel_err = float((torch.linalg.vector_norm(x_holdout - y_hold, dim=-1)
                 / torch.linalg.vector_norm(x_holdout, dim=-1)).mean())

print("=== held-out reconstruction ===")
print(f"plain AE (d={LATENT_DIM}) mse_per_dim = {stats[\'mse_per_dim\']:.6f}")
print(f"plain AE mean relative error         = {rel_err:.4f}  ({100 * rel_err:.1f}% of point norm)")

fig = plt.figure(figsize=(11, 9))
for i, (data, title) in enumerate([(X, "original"), (y_all, "plain AE reconstruction")]):
    ax = fig.add_subplot(2, 2, i + 1, projection="3d")
    ax.scatter(data[:, 0], data[:, 1], data[:, 2], c=t, cmap="viridis", s=4)
    ax.view_init(**VIEW); ax.set_title(f"{title} (colour = t)")
    ax.set_xlim(X[:, 0].min(), X[:, 0].max()); ax.set_ylim(X[:, 1].min(), X[:, 1].max())
    ax.set_zlim(X[:, 2].min(), X[:, 2].max())
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax2 = fig.add_subplot(2, 2, i + 3)
    ax2.scatter(data[:, 0], data[:, 2], c=t, cmap="viridis", s=4)
    ax2.set_title(f"{title} \u2014 x-z plane"); ax2.set_aspect("equal")
    ax2.set_xlim(X[:, 0].min(), X[:, 0].max()); ax2.set_ylim(X[:, 2].min(), X[:, 2].max())
    ax2.set_xlabel("x"); ax2.set_ylabel("z")
plt.tight_layout(); plt.show()


## §5-§6. Curvature through the plain decoder, and against the analytic answer

`decoder_curvature.plain_decoder_curvature` takes **latent** coordinates, not ambient points, so
the notebook encodes first: `z_all = model.encode(x_all.double())`. `model.decode`, never the
round-trip `forward` method, is the map differentiated, because differentiating the round-trip
map would measure the curvature of the encoder-composed manifold instead of the decoder's own
image manifold.

The model is cast to float64 first (`model.double()`): second derivatives are exactly where
float32 noise shows, and `plain_decoder_curvature`'s own float64 guard refuses anything else.

The four axes -- direction, magnitude (median and CV), calibration, rank -- are computed by
`chart_curvature.curvature_fidelity_report` and `curvature_probe.spearman_gate_statistic`
unchanged, never reimplemented. They are reported strictly against the raw-point comparator
recomputed on the identical fixture in this same run, per `02.6-SCREENING-RULE.md`; no absolute
bar of any kind is used anywhere in this notebook.

In [ ]:
model.double()  # second derivatives are exactly where float32 noise shows; the module refuses float32
with torch.no_grad():
    z_all = model.encode(x_all.double())

curv0 = time.time()
out = decoder_curvature.plain_decoder_curvature(model, z_all)
curv_wall = time.time() - curv0

H_plain = out["H_vec"].numpy()
h_plain = out["H_norm"].numpy()
cond = out["metric_condition_number"].numpy()
cond_max = float(cond.max())

base0 = time.time()
H_raw = curvature_probe.centroid_mean_curvature(X, k=K_BASELINE, d=LATENT_DIM)
h_raw = curvature_probe.mean_curvature_norm(H_raw)
base_wall = time.time() - base0

rho_plain = curvature_probe.spearman_gate_statistic(h_plain, H_norm)
rho_raw = curvature_probe.spearman_gate_statistic(h_raw, H_norm)
fid_plain = chart_curvature.curvature_fidelity_report(H_plain, H_true)
fid_raw = chart_curvature.curvature_fidelity_report(H_raw, H_true)

print(f"plain-decoder field: {curv_wall:.1f} s   (raw-point comparator {base_wall:.1f} s)")
print(f"metric condition number: median {np.median(cond):.2f}  max {cond_max:.2f}")
print(f"n_excluded: plain decoder {fid_plain[\'n_excluded\']}   raw points {fid_raw[\'n_excluded\']}")
print()
hdr = f"{\'\':<24}{\'plain decoder\':>16}{\'raw points (k=30)\':>20}"
print(hdr); print("-" * len(hdr))
for label, a, b in [
    ("1. direction  cos", fid_plain["median_cosine_similarity"], fid_raw["median_cosine_similarity"]),
    ("2. magnitude  median", fid_plain["median_magnitude_ratio"], fid_raw["median_magnitude_ratio"]),
    ("2. magnitude  CV", fid_plain["magnitude_ratio_cv"], fid_raw["magnitude_ratio_cv"]),
    ("3. calib. slope a", fid_plain["calibration_slope"], fid_raw["calibration_slope"]),
    ("3. calib. intercept b", fid_plain["calibration_intercept"], fid_raw["calibration_intercept"]),
    ("3. calib. R^2", fid_plain["calibration_r2"], fid_raw["calibration_r2"]),
    ("4. rank  Spearman", rho_plain, rho_raw),
]:
    print(f"{label:<24}{a:>16.4f}{b:>20.4f}")


### The pictures

**Each panel below is normalized to its own 2-98 percentile range on purpose**, so what is being
compared is *spatial pattern* only -- amplitude is deliberately not readable from these colours,
because amplitude belongs to the magnitude-ratio and calibration numbers above, and conflating
the two is exactly the error this notebook exists to avoid.

Then the two identity-line scatters, on shared axes, is where amplitude *is* readable: points on
the diagonal mean the estimate has the right size, a flattened cloud means the rank statistic has
nothing to work with, and a systematically shallow slope is the smoothing failure mode.

In [ ]:
panels = [("analytic ||H||", H_norm), ("plain-decoder ||H||", h_plain),
          ("raw-point ||H|| (k=30)", h_raw)]

fig = plt.figure(figsize=(15, 8.5))
for j, (title, vals) in enumerate(panels):
    lo, hi = np.percentile(vals, [2, 98])
    ax = fig.add_subplot(2, 3, j + 1, projection="3d")
    ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=vals, cmap="magma", s=4, vmin=lo, vmax=hi)
    ax.view_init(**VIEW); ax.set_title(f"{title}\n(own 2-98 pct scale)", fontsize=10)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax2 = fig.add_subplot(2, 3, j + 4)
    sc = ax2.scatter(X[:, 0], X[:, 2], c=vals, cmap="magma", s=4, vmin=lo, vmax=hi)
    ax2.set_title(f"{title} \u2014 x-z plane", fontsize=10); ax2.set_aspect("equal")
    ax2.set_xlabel("x"); ax2.set_ylabel("z")
    fig.colorbar(sc, ax=ax2, fraction=0.046)
plt.tight_layout(); plt.show()

lim = (0.0, float(max(H_norm.max(), np.percentile(h_plain, 99), np.percentile(h_raw, 99))) * 1.05)
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for ax, (name, vals, rho) in zip(axes, [("plain decoder", h_plain, rho_plain),
                                        ("raw points (k=30)", h_raw, rho_raw)]):
    ax.scatter(H_norm, vals, s=4, alpha=0.35)
    ax.plot(lim, lim, "k--", lw=1, label="identity")
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
    ax.set_xlabel("analytic ||H||"); ax.set_ylabel("estimated ||H||")
    ax.set_title(f"{name}   (Spearman {rho:.4f})")
    ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()


## §7. Read-out

**The pattern to look for**, following `02.5_swiss_roll_chart_curvature_check.ipynb`'s own
precedent, is whether a PASS on reconstruction sits beside a FAIL on curvature -- that
combination is exactly D-09's counter-risk showing up on a manifold where the answer is known,
and it is invisible on data with no known answer.

The four axes below are reported **separately and never combined into one number**; a rank
statistic alone cannot see a decoder that compresses every magnitude by a constant -- it would
score that decoder a perfect Spearman while badly misreporting amplitude, which is exactly why
magnitude and calibration are reported too.

In [ ]:
dir_ok = fid_plain["median_cosine_similarity"] >= fid_raw["median_cosine_similarity"]
mag_ratio_ok = abs(fid_plain["median_magnitude_ratio"] - 1.0) <= abs(fid_raw["median_magnitude_ratio"] - 1.0)
mag_cv_ok = fid_plain["magnitude_ratio_cv"] <= fid_raw["magnitude_ratio_cv"]
mag_ok = mag_ratio_ok and mag_cv_ok
calib_ok = abs(fid_plain["calibration_slope"] - 1.0) <= abs(fid_raw["calibration_slope"] - 1.0)
rank_ok = rho_plain >= rho_raw

print(f"1. direction   : {dir_ok}   (plain median cosine {fid_plain[\'median_cosine_similarity\']:.4f} "
      f"vs raw {fid_raw[\'median_cosine_similarity\']:.4f})")
print(f"2. magnitude   : {mag_ok}   "
      f"(|ratio-1| plain {abs(fid_plain[\'median_magnitude_ratio\'] - 1.0):.4f} vs raw "
      f"{abs(fid_raw[\'median_magnitude_ratio\'] - 1.0):.4f};  CV plain {fid_plain[\'magnitude_ratio_cv\']:.4f} "
      f"vs raw {fid_raw[\'magnitude_ratio_cv\']:.4f})")
print(f"3. calibration : {calib_ok}   "
      f"(|slope-1| plain {abs(fid_plain[\'calibration_slope\'] - 1.0):.4f} vs raw "
      f"{abs(fid_raw[\'calibration_slope\'] - 1.0):.4f};  plain slope {fid_plain[\'calibration_slope\']:.4f} "
      f"intercept {fid_plain[\'calibration_intercept\']:.4f} R^2 {fid_plain[\'calibration_r2\']:.4f})")
print(f"4. rank        : {rank_ok}   (plain Spearman {rho_plain:.4f} vs raw {rho_raw:.4f})")
print()
print("These four are separate sanity read-outs, are never combined into one score, and gate nothing")
print("on their own -- the phase's screening bars are applied in 02.6-FINDINGS.md, not here.")
print()

read_out = (
    f"Read-out: the plain auto-encoder reconstructs the Swiss roll at {100 * rel_err:.1f}% relative "
    f"error, and curvature taken exactly through that trained decoder scores direction "
    f"{\'at least as well as\' if dir_ok else \'worse than\'} the raw-point comparator "
    f"(cosine {fid_plain[\'median_cosine_similarity\']:.3f} vs {fid_raw[\'median_cosine_similarity\']:.3f}), "
    f"magnitude {\'no worse than\' if mag_ok else \'worse than\'} it "
    f"(median ratio {fid_plain[\'median_magnitude_ratio\']:.3f} at CV {fid_plain[\'magnitude_ratio_cv\']:.3f} "
    f"vs raw {fid_raw[\'median_magnitude_ratio\']:.3f} at CV {fid_raw[\'magnitude_ratio_cv\']:.3f}), "
    f"calibration {\'no worse than\' if calib_ok else \'worse than\'} it "
    f"(slope {fid_plain[\'calibration_slope\']:.3f} vs raw {fid_raw[\'calibration_slope\']:.3f}), "
    f"and rank {\'beats\' if rank_ok else \'loses to\'} it "
    f"(Spearman {rho_plain:.4f} vs {rho_raw:.4f})."
)
print(read_out)
print()
print(f"(context, not a fifth axis) metric condition number: median {np.median(cond):.2f}  max {cond_max:.2f}")
